# SenGuard-MSC · Notebook 03 — Independent MSC evaluation (G3)

Applies scores frozen in notebook 02 to GSE309666 (AD-MSC, primary) and GSE178514 (BM-MSC, supporting). It verifies hashes first and stops if anything changed after the freeze.

**What this notebook does not do:** there is no AUROC, accuracy, threshold or calibration against passage, because no per-sample senescence label exists. Everything here is a within-donor trend. Scores are model inference; passage and donor are observations. Nothing here measures potency, immune function or safety.

In [ ]:
import json, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
ROOT_GUESS = (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, DEV, ASC, BM
import joblib
from scipy import stats
import matplotlib.pyplot as plt
sg.set_fig_style()
np.random.seed(sg.SEED)

freeze = json.loads((RESULTS / 'frozen_model.json').read_text())
if freeze['protocol_sha256'] != sg.sha256(REPORTS / 'ANALYSIS_PROTOCOL.md'):
    raise RuntimeError('Protocol changed after the freeze.')
for name, art in freeze['artifacts'].items():
    if sg.sha256(art['path']) != art['sha256']:
        raise RuntimeError(f'{name} changed after the freeze.')
print('All frozen artifacts verified:', list(freeze['artifacts']))
print('Chosen model:', freeze['chosen_model'], freeze['params'], '| frozen', freeze['frozen_utc'])

manifest = sg.load_manifest()
asc_meta = manifest[(manifest.study_accession == ASC) & manifest.use_in_mrna_analysis].copy()
bm_meta = manifest[(manifest.study_accession == BM) & manifest.use_in_mrna_analysis].copy()
asc_meta['subject'], bm_meta['subject'] = asc_meta.donor, bm_meta.candidate_biological_id.astype(int).astype(str)

asc_raw, asc_ann = sg.load_as_deposited(ASC)
asc_norm = pd.read_csv(PROC / f'{ASC}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[asc_meta.index]
bm_raw, _ = sg.load_as_deposited(BM)
bm_tpm = pd.read_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t', index_col=0).T.loc[bm_meta.index]
idmap = pd.read_csv(PROC / f'{BM}_gene_id_map_final.tsv', sep='\t', dtype=str).set_index('gene_id')
print('AD-MSC', asc_norm.shape, 'donors', asc_meta.subject.nunique(), '| BM-MSC', bm_tpm.shape, 'putative donors', bm_meta.subject.nunique())

asc_sym = sg.to_symbols(asc_norm, asc_ann.rename(columns={'symbol': 'symbol'}))
bm_tpm_all = sg.tpm_from_fpkm(bm_raw.loc[bm_meta.index])
bm_sym = sg.to_symbols(bm_tpm_all, idmap)
print('Symbol matrices', asc_sym.shape, bm_sym.shape)

## Compute frozen scores
Higher means more senescence-like for every score, using the direction fixed in notebook 02. Absent markers are skipped, as in the authors' script, and coverage is recorded. GSE178514 uses TPM rescaled from FPKM rather than the normalized counts the score was built on, which is a known limit. Its within-donor change is interpretable; its absolute value is not comparable to SenCat.

In [ ]:
h, extra = sg.load_gene_sets()
pub = sg.load_sencat_markers().coef
weights = {'SenCat published': pub}
for tag in ['all14', 'no_BMMSC', 'no_BMMSC_prAD']:
    weights[f'SenCat refit {tag}'] = pd.read_csv(RESULTS / f'frozen_weights_refit_{tag}.csv').set_index('gene_id').coef
prolif = set(h['HALLMARK_E2F_TARGETS']) | set(h['HALLMARK_G2M_CHECKPOINT'])
sym_of = pd.read_csv(sg.EXT / 'wf-ml-markers-senescence/analysis/transcriptomics_loose5000f_tuned_common_features.csv').set_index('converted_alias').gene
pub_noprolif = pub[~sym_of.reindex(pub.index).isin(prolif)]
weights['SenCat published, no E2F/G2M genes'] = pub_noprolif
print('Published markers removed as E2F/G2M genes:', len(pub) - len(pub_noprolif))

UP = 'CACNB4 CRYAB DAW1 EFEMP1 EPAS1 GALNT5 H1-2 KRT34 MELTF MEST MFGE8 MGARP MOXD1 SERPINB2 SERPINE2 SLC43A2 SRGN SVEP1 SYNPO2 TMEM178B'.split()
DOWN = 'AURKB BUB1 BUB1B CDC20 DEPDC1 KIF14 KIF18B KIF2C KIFC1 KNL1 LMNB1 NCAPH NUF2 PIMREG PKMYT1 SKA3 TNC TROAP TTK UBE2C'.split()

canon = pd.read_csv(sg.EXT / 'wf-ml-markers-senescence/config/Canonical_gene_markers.csv', index_col=0).gene
CANON_SYM = {'p16': 'CDKN2A', 'p21': 'CDKN1A', 'IL6': 'IL6', 'GDF15': 'GDF15', 'LMNB1': 'LMNB1'}

dev_counts, _ = sg.load_as_deposited(DEV)
labels = pd.read_csv(PROC / f'{DEV}_labels.csv').set_index('sample_id')
dev_counts = dev_counts.loc[labels.index]

def model_decision(tag, ext_values):
    art = joblib.load(freeze['artifacts'][f"{freeze['chosen_model']}_{tag}"]['path'])
    mask = np.ones(len(labels), bool) if tag == 'all14' else ~labels.cell_line.isin(['BMMSC'] if tag == 'no_BMMSC' else ['BMMSC', 'prAD']).to_numpy()
    train = sg.fold_transform(dev_counts.loc[mask], art['genes'], art['ref'])
    means = np.expm1(train).mean()
    shared_ref = [g for g in art['ref'].index if g in ext_values.columns]
    sf = sg.mor_size_factors(ext_values[shared_ref], art['ref'].loc[shared_ref])
    x = pd.DataFrame({g: (ext_values[g] / sf if g in ext_values.columns else pd.Series(means[g], index=ext_values.index)) for g in art['genes']})[art['genes']]
    cover = float(np.mean([g in ext_values.columns for g in art['genes']]))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return pd.Series(sg.decision_values(art['model'], np.log1p(x).to_numpy()), index=ext_values.index), cover

def all_scores(norm, sym, tag_for_model):
    out, cov = {}, {}
    for name, w in weights.items():
        s, n = sg.weighted_log1p_score(norm, w)
        out[name], cov[name] = s, n / len(w)
    husi = sg.load_husi_weights()
    out['hUSI raw'], n = sg.husi_raw(sym, husi)
    cov['hUSI raw'] = n / len(husi)
    for name, genes, sign in [('SenMayo', extra['SenMayo'], 1), ('Hallmark E2F (inverted)', h['HALLMARK_E2F_TARGETS'], -1), ('Hallmark G2M (inverted)', h['HALLMARK_G2M_CHECKPOINT'], -1)]:
        s, n = sg.rank_set_score(sym, genes)
        out[name], cov[name] = sign * s, n / len(genes)
    su, nu = sg.rank_set_score(sym, UP)
    sd, nd = sg.rank_set_score(sym, DOWN)
    out['AD-MSC 40 biomarkers (up minus down)'], cov['AD-MSC 40 biomarkers (up minus down)'] = su - sd, (nu + nd) / 40
    for key, gene in CANON_SYM.items():
        if gene in sym.columns:
            v = np.log1p(sym[gene])
            out[f'{gene} transcript' + (' (inverted)' if gene == 'LMNB1' else '')] = -v if gene == 'LMNB1' else v
            cov[f'{gene} transcript' + (' (inverted)' if gene == 'LMNB1' else '')] = 1.0
    m, c = model_decision(tag_for_model, norm)
    label = f"Frozen {freeze['chosen_model']}, target lineage excluded, decision value"
    out[label], cov[label] = m, c
    return pd.DataFrame(out), cov

asc_scores, asc_cov = all_scores(asc_norm, asc_sym, 'no_BMMSC_prAD')
bm_scores, bm_cov = all_scores(bm_tpm, bm_sym, 'no_BMMSC')
coverage = pd.DataFrame({'GSE309666': asc_cov, 'GSE178514': bm_cov}).round(3).rename_axis('score')
coverage.to_csv(RESULTS / 'external_score_coverage.csv')
print(coverage.to_string())

long = pd.concat([
    asc_scores.join(asc_meta[['study_accession', 'subject', 'passage']]).rename_axis('sample_id').reset_index().melt(['sample_id', 'study_accession', 'subject', 'passage'], var_name='score_name', value_name='score'),
    bm_scores.join(bm_meta[['study_accession', 'subject', 'passage']]).rename_axis('sample_id').reset_index().melt(['sample_id', 'study_accession', 'subject', 'passage'], var_name='score_name', value_name='score')])
long['evidence'] = 'model inference on measured expression; passage and donor from GEO'
long.to_csv(RESULTS / 'external_predictions.csv', index=False)
print('external_predictions.csv rows', len(long))

## Primary result: published SenCat score across passage, within donor [measured]
Slope is per passage. Sign-flip p cannot go below 0.0625 with five donors or 0.25 with three, as the protocol states up front.

In [ ]:
def trend_table(scores, meta):
    rows, per = [], {}
    for name in scores.columns:
        t = sg.per_donor_trend(scores[name], meta.passage, meta.subject)
        per[name] = t
        rows.append({'score': name, 'mean_slope': t.slope_per_passage.mean(), 'donors_positive': int((t.slope_per_passage > 0).sum()), 'donors': len(t),
                     'median_rho': t.spearman_rho.median(), 'sign_flip_p': sg.exact_sign_flip_p(t.slope_per_passage), 'mean_last_minus_first': t.last_minus_first.mean(),
                     'standardized_slope': t.slope_per_passage.mean() / scores[name].std()})
    return pd.DataFrame(rows), per

asc_table, asc_per = trend_table(asc_scores, asc_meta)
bm_table, bm_per = trend_table(bm_scores, bm_meta)
asc_table.to_csv(RESULTS / 'external_trends_GSE309666.csv', index=False)
bm_table.to_csv(RESULTS / 'external_trends_GSE178514.csv', index=False)
print('AD-MSC per donor, published SenCat:')
print(asc_per['SenCat published'].round(3).to_string(index=False))
print('\nBM-MSC per suffix, published SenCat:')
print(bm_per['SenCat published'].round(3).to_string(index=False))
print('\nAll scores, AD-MSC:')
print(asc_table.round(3).to_string(index=False))
print('\nAll scores, BM-MSC:')
print(bm_table.round(3).to_string(index=False))

## Is the trend specific to this signature? Competitive null
Late-passage cells change a great many genes, so a rising score can happen for a random signature too. These draws ask whether the actual weights rise more than shuffled weights on the same genes, or random expressed genes with the same weights.

In [ ]:
nulls = {}
rows = []
for cohort, norm, meta, tags in [('GSE309666', asc_norm, asc_meta, ['SenCat published', 'SenCat refit no_BMMSC_prAD']), ('GSE178514', bm_tpm, bm_meta, ['SenCat published', 'SenCat refit no_BMMSC'])]:
    logx = np.log1p(norm)
    universe = logx.columns[logx.mean() > 1]
    for name in tags:
        for mode in ['weights', 'genes']:
            obs, null, p = sg.competitive_null(logx, weights[name], meta.passage, meta.subject, n=1000, mode=mode, universe=universe)
            nulls[(cohort, name, mode)] = (obs, null, p)
            rows.append({'cohort': cohort, 'score': name, 'null': mode, 'observed_mean_slope': obs, 'null_mean': null.mean(), 'null_sd': null.std(), 'empirical_two_sided_p': p, 'z': (obs - null.mean()) / null.std()})
null_table = pd.DataFrame(rows)
null_table.to_csv(RESULTS / 'external_competitive_null.csv', index=False)
print(null_table.round(4).to_string(index=False))

## Sensitivity analyses
- GSE309666 without the 18th record (GSM9271440).
- Normalization alternatives, using primary score weights.
- Early (p1 to p3) versus later change. The paper describes stable proliferation through at least p3, so this is a direct comparison with a published observation.
- Published statement that donor B already scored high at p3.

In [ ]:
sens = []
keep = asc_meta.index != 'GSM9271440'
t = sg.per_donor_trend(asc_scores['SenCat published'][keep], asc_meta.passage[keep], asc_meta.subject[keep])
sens.append({'analysis': 'GSE309666 excluding GSM9271440', 'mean_slope': t.slope_per_passage.mean(), 'donors_positive': int((t.slope_per_passage > 0).sum()), 'donors': len(t)})
asc_raw_s = asc_raw.loc[asc_meta.index]
for label, x in [('GSE309666 as deposited, no size factors', asc_raw_s), ('GSE309666 scaled to 1e6 per sample', sg.scale_to_million(asc_raw_s))]:
    s, _ = sg.weighted_log1p_score(x, pub)
    t = sg.per_donor_trend(s, asc_meta.passage, asc_meta.subject)
    sens.append({'analysis': label, 'mean_slope': t.slope_per_passage.mean(), 'donors_positive': int((t.slope_per_passage > 0).sum()), 'donors': len(t)})
dev_ref = sg.mor_reference(sg.author_gene_filter(dev_counts))
shared_ref = asc_raw_s.columns.intersection(dev_ref.index)
sfd = sg.mor_size_factors(asc_raw_s[shared_ref], dev_ref[shared_ref])
s, _ = sg.weighted_log1p_score(asc_raw_s.div(sfd, axis=0), pub)
t = sg.per_donor_trend(s, asc_meta.passage, asc_meta.subject)
sens.append({'analysis': 'GSE309666 size factors from the SenCat reference', 'mean_slope': t.slope_per_passage.mean(), 'donors_positive': int((t.slope_per_passage > 0).sum()), 'donors': len(t)})
bm_raw_ens = bm_raw.loc[bm_meta.index]
wm = idmap.ensembl_final.reindex(bm_raw_ens.columns)
bm_fpkm_ens = bm_raw_ens.loc[:, wm.notna()].T.groupby(wm.dropna().values).sum().T
s, _ = sg.weighted_log1p_score(bm_fpkm_ens, pub)
t = sg.per_donor_trend(s, bm_meta.passage, bm_meta.subject)
sens.append({'analysis': 'GSE178514 FPKM as deposited', 'mean_slope': t.slope_per_passage.mean(), 'donors_positive': int((t.slope_per_passage > 0).sum()), 'donors': len(t)})
sens = pd.DataFrame(sens)
sens.to_csv(RESULTS / 'external_sensitivity.csv', index=False)
print(sens.round(3).to_string(index=False))

phase = []
for d, g in asc_scores['SenCat published'].groupby(asc_meta.subject):
    p = asc_meta.loc[g.index].passage
    phase.append({'donor': d, 'p1': g[p == 1].iloc[0], 'p3': g[p == 3].iloc[0], 'final_passage': int(p.max()), 'final': g[p == p.max()].iloc[0],
                  'p1_to_p3': g[p == 3].iloc[0] - g[p == 1].iloc[0], 'p3_to_final': g[p == p.max()].iloc[0] - g[p == 3].iloc[0], 'published_growth_status': asc_meta.loc[g.index].published_donor_growth_status.iloc[0]})
phase = pd.DataFrame(phase)
phase.to_csv(RESULTS / 'external_phase_changes_GSE309666.csv', index=False)
print('\nPublished SenCat score by phase, per donor:')
print(phase.round(3).to_string(index=False))
sm = asc_scores['SenMayo'].to_frame('SenMayo').join(asc_meta[['subject', 'passage']]).pivot(index='subject', columns='passage', values='SenMayo')
print('\nSenMayo rank score by donor and passage. The paper reports GSVA SenMayo rising, with donor B as high at p3 as at p8; ours is a different scoring method:')
print(sm.round(3).to_string())

## Training overlap in hUSI [measured]
Notebook 02 found that the hUSI training metadata mentions GSE178514. Those rows are listed here, because a score trained on a cohort can't be independently evaluated on it. Passing this through doesn't change any frozen score: it decides how the hUSI row is read.

In [ ]:
husi_meta = pd.read_csv(sg.EXT / 'HUSI/Data/trainSet_meta843.csv', low_memory=False, encoding='latin-1')
husi_hits = {acc: husi_meta[husi_meta.astype(str).agg(' '.join, axis=1).str.contains(acc)] for acc in [DEV, ASC, BM]}
print({k: len(v) for k, v in husi_hits.items()})
print(husi_hits[BM].dropna(axis=1, how='all').iloc[:, :12].to_string())
msc_rows = husi_meta[husi_meta.astype(str).agg(' '.join, axis=1).str.contains(r'mesenchym|\bMSC', case=False, regex=True)]
print('MSC-related training rows:', len(msc_rows), '| studies:', msc_rows.iloc[:, 1].value_counts().to_dict())
husi_overlap_note = f"hUSI training metadata lists {len(husi_hits[BM])} rows from GSE178514, so hUSI on GSE178514 is NOT an independent test. It lists {len(husi_hits[ASC])} rows from GSE309666 and {len(husi_hits[DEV])} from GSE284666, and {len(msc_rows)} MSC-related rows overall."
print(husi_overlap_note)
husi_hits[BM].to_csv(RESULTS / 'husi_training_overlap_GSE178514.csv', index=False)

## Domain shift between SenCat and the MSC cohorts [measured]
No external data is used to fit anything: PCA is fit on SenCat and external samples are only projected. Nearest-neighbour distance is compared with how far SenCat cell types sit from one another when each is held out.

In [ ]:
from sklearn.decomposition import PCA
dev_filt = sg.author_gene_filter(dev_counts)
dev_norm, _, ref = sg.mor_normalize(dev_filt)
dev_log = np.log1p(dev_norm)
shared_all = [g for g in pub.index if g in dev_log.columns and g in asc_norm.columns and g in bm_tpm.columns]
pca = PCA(n_components=10, random_state=sg.SEED).fit(dev_log[shared_all])
pz = pd.DataFrame(pca.transform(dev_log[shared_all]), index=dev_log.index)
ext = {'GSE309666': np.log1p(asc_norm[shared_all]), 'GSE178514': np.log1p(sg.scale_to_million(bm_tpm)[shared_all])}
loo = []
for ct in labels.cell_line.unique():
    a, b = pz[labels.cell_line == ct].to_numpy(), pz[labels.cell_line != ct].to_numpy()
    loo.append(np.sqrt(((a[:, None] - b[None]) ** 2).sum(-1)).min(1))
ref_dist = np.concatenate(loo)
rows = []
for cohort, x in ext.items():
    z = pca.transform(x)
    dist = np.sqrt(((z[:, None] - pz.to_numpy()[None]) ** 2).sum(-1))
    nn = labels.cell_line.to_numpy()[dist.argmin(1)]
    rows.append({'cohort': cohort, 'genes_shared': len(shared_all), 'median_nn_distance': float(np.median(dist.min(1))), 'dev_held_out_type_p95': float(np.percentile(ref_dist, 95)),
                 'share_beyond_p95': float((dist.min(1) > np.percentile(ref_dist, 95)).mean()), 'nearest_cell_types': pd.Series(nn).value_counts().to_dict(),
                 'spearman_gene_means_vs_BMMSC': float(stats.spearmanr(x.mean(), dev_log.loc[labels.cell_line == 'BMMSC', shared_all].mean()).statistic),
                 'spearman_gene_means_vs_prAD': float(stats.spearmanr(x.mean(), dev_log.loc[labels.cell_line == 'prAD', shared_all].mean()).statistic)})
domain = pd.DataFrame(rows)
domain.to_csv(RESULTS / 'external_domain_shift.csv', index=False)
print(domain.round(3).to_string(index=False))

pub_dev, _ = sg.weighted_log1p_score(dev_norm, pub)
ref_scores = pub_dev.groupby([labels.cell_line, labels.y]).mean().unstack().rename(columns={0: 'control_mean', 1: 'senescent_mean'})
print('\nPublished score in SenCat, by type (reference only; do not threshold across platforms):')
print(ref_scores.loc[['BMMSC', 'prAD', 'BJ', 'WI-38']].round(2).to_string())
print('\nAD-MSC score range %.2f to %.2f' % (asc_scores['SenCat published'].min(), asc_scores['SenCat published'].max()))

## Figures
**Figure 2:** per-donor trajectories, one panel per donor, as model inference on measured expression. **Figure 3:** direction across scores and donors. **Figure 4:** the competitive null. Tables behind them are in `results/`.

In [ ]:
def traj(axs, scores, meta, per, title_prefix):
    donors = sorted(meta.subject.unique())
    for ax, d in zip(axs, donors):
        g = meta[meta.subject == d].sort_values('passage')
        ax.plot(g.passage, scores.loc[g.index], color=sg.PAL['blue'], marker='o', ms=6, mec=sg.PAL['surface'], mew=1.5)
        sl = per[per.donor == d].slope_per_passage.iloc[0]
        ax.set_title(f'{title_prefix} {d}', fontsize=10)
        ax.text(0.04, 0.92, f'slope {sl:+.2f}/passage', transform=ax.transAxes, fontsize=8.5, color=sg.PAL['ink2'])
        ax.set_xlabel('Passage')
        ax.grid(axis='x', visible=False)

fig, axes = plt.subplots(2, 5, figsize=(13, 5.6), gridspec_kw={'hspace': 0.5, 'wspace': 0.12})
for ax in axes[0][1:]:
    ax.sharey(axes[0][0]); ax.tick_params(labelleft=False)
traj(axes[0], asc_scores['SenCat published'], asc_meta, asc_per['SenCat published'], 'AD-MSC donor')
axes[0][0].set_ylabel('SenCat score (published)')
for ax in axes[1][1:3]:
    ax.sharey(axes[1][0]); ax.tick_params(labelleft=False)
traj(axes[1][:3], bm_scores['SenCat published'], bm_meta, bm_per['SenCat published'], 'BM-MSC id')
axes[1][0].set_ylabel('SenCat score (published)')
for ax in axes[1][3:]:
    ax.axis('off')
axes[1][3].text(0, 0.82, 'Model inference on measured expression.\nPassage is not a senescence label.\nBM-MSC comes from FPKM, so its scale\nis not comparable with AD-MSC.', fontsize=9.5, color=sg.PAL['ink2'], va='top')
fig.savefig(sg.FIGS / 'fig2_donor_trajectories.png', bbox_inches='tight')
plt.show()

order = list(asc_scores.columns) + [c for c in bm_scores.columns if c not in asc_scores.columns]
def rho_frame(per):
    return pd.DataFrame({k: v.set_index('donor').spearman_rho for k, v in per.items()}).T.reindex(order)
ra, rb = rho_frame(asc_per), rho_frame(bm_per)
ra.columns = [f'AD {c}' for c in ra.columns]
rb.columns = [f'BM {c}' for c in rb.columns]
mat = pd.concat([ra, rb], axis=1)
mat.to_csv(RESULTS / 'external_rho_heatmap_data.csv')
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#e6e5df', '#eb6834'])
fig, ax = plt.subplots(figsize=(9.5, 0.36 * len(mat) + 1.4))
im = ax.imshow(mat.to_numpy(dtype=float), cmap=cmap, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=45, ha='right', fontsize=8.5)
ax.set_yticks(range(mat.shape[0]), mat.index, fontsize=8.5)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xticks(np.arange(-.5, mat.shape[1]), minor=True); ax.set_yticks(np.arange(-.5, mat.shape[0]), minor=True)
ax.grid(which='minor', color=sg.PAL['surface'], linewidth=2); ax.tick_params(which='minor', length=0)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.iat[i, j]
        ax.text(j, i, '' if pd.isna(v) else f'{v:+.1f}', ha='center', va='center', fontsize=7.5, color=sg.PAL['ink'])
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02); cb.set_label('Spearman rho with passage'); cb.outline.set_visible(False)
ax.set_title('Direction with passage, per donor (3-4 points each, so rho is coarse)')
fig.savefig(sg.FIGS / 'fig3_direction_by_score.png', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
for ax, cohort, label in [(axes[0], 'GSE309666', 'AD-MSC'), (axes[1], 'GSE178514', 'BM-MSC')]:
    obs, null, p = nulls[(cohort, 'SenCat published', 'genes')]
    ax.hist(null, bins=40, color=sg.PAL['muted'], alpha=0.75, label='Random expressed genes, shuffled weights')
    ax.axvline(obs, color=sg.PAL['orange'], lw=2, label=f'Published weights (p = {p:.3f})')
    ax.set_title(label); ax.set_xlabel('Mean within-donor slope per passage'); ax.set_ylabel('Draws'); ax.legend(fontsize=8, loc='center', bbox_to_anchor=(0.55, 0.6)); ax.grid(axis='x', visible=False)
fig.tight_layout()
fig.savefig(sg.FIGS / 'fig4_competitive_null.png')
plt.show()

## G3 report

In [ ]:
A = asc_table[asc_table.score == 'SenCat published'].iloc[0]
B = bm_table[bm_table.score == 'SenCat published'].iloc[0]
report = f"""# G3: external MSC evaluation

Scores frozen in notebook 02 (frozen {freeze['frozen_utc']}), hashes verified at load. No labels exist for these cohorts, so no classification metric was computed.

## Primary: published SenCat score, GSE309666 [measured]
Mean slope {A.mean_slope:+.3f} per passage; {A.donors_positive}/{A.donors} donors positive; median rho {A.median_rho:.2f}; exact sign-flip p {A.sign_flip_p:.4f} (smallest possible 0.0625).

## Supporting: GSE178514 [measured]
Mean slope {B.mean_slope:+.3f} per passage; {B.donors_positive}/{B.donors} ids positive; median rho {B.median_rho:.2f}; exact sign-flip p {B.sign_flip_p:.4f} (smallest possible 0.25).

## Competitive null [measured]
{null_table.round(4).to_string(index=False)}

## All scores [measured]
AD-MSC:
{asc_table.round(3).to_string(index=False)}

BM-MSC:
{bm_table.round(3).to_string(index=False)}

## Sensitivity [measured]
{sens.round(3).to_string(index=False)}

## Domain shift [measured]
{domain.round(3).to_string(index=False)}

## Limits
- Passage and donor are observations; every score is inference. Growth-arrest status exists only per donor at final passage, and D and E are censored at passage 10.
- Published weights were fit with BMMSC in the data. The refit no_BMMSC and no_BMMSC_prAD rows are the cell-type-independent versions.
- {husi_overlap_note}
- GSE178514 is FPKM and its donor mapping is inferred. Culture systems differ (platelet lysate vs FBS), and so do library types (polyA vs rRNA-depleted).
- The 40 AD-MSC biomarkers came from GSE309666, so their result there is circular; GSE178514 is their only independent test here.
- No conclusion on potency, immune function or clinical safety follows from these data.
"""
(REPORTS / 'G3_EXTERNAL_VALIDATION.md').write_text(report, encoding='utf-8')
print(report)
outs = [RESULTS / f for f in ['external_predictions.csv', 'external_score_coverage.csv', 'external_trends_GSE309666.csv', 'external_trends_GSE178514.csv', 'external_competitive_null.csv', 'external_sensitivity.csv', 'external_phase_changes_GSE309666.csv', 'external_domain_shift.csv', 'external_rho_heatmap_data.csv']] + [sg.FIGS / f for f in ['fig2_donor_trajectories.png', 'fig3_direction_by_score.png', 'fig4_competitive_null.png']] + [REPORTS / 'G3_EXTERNAL_VALIDATION.md']
print(sg.output_manifest(outs).assign(path=lambda d: d.path.str.replace(str(ROOT), '', regex=False))[['path', 'bytes']].to_string())